In [ ]:
from typing import List, Dict
from pydantic import BaseModel,Field
from langchain.agents import create_agent
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama
# 1. Define the output schema for parsing


questions = [
{"name": "impact_climate_biodiversity", "question": "This papers talks about the impact of climate change on biodiversity"},
{"name": "impact_climate_human_scoiety", "question": "This papers talks about the impact of climate change on human society"}
]
    
questions_str = "\n".join(
    [f"{q['name']}: {q['question']}" for q in questions]
)


In [73]:
class QuestionAnswer(BaseModel):
    answer: bool = Field(description="Binary anwser of the assessment") # True of False
    explanation: str = Field(description="Explanation of the binary answer in one or two sentences")  # 1-2 sentences


parser = PydanticOutputParser(pydantic_object=QuestionAnswer)

system_prompt = f"""
You are a precise assistant.
For the following assessment, answer with a JSON object containing:
- 'answer': 'True' or 'False'
- 'explanation': a short explanation (1-2 sentences)

Format: {parser.get_format_instructions()}
"""
print(system_prompt)


You are a precise assistant.
For the following assessment, answer with a JSON object containing:
- 'answer': 'True' or 'False'
- 'explanation': a short explanation (1-2 sentences)

Format: The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"answer": {"description": "Binary anwser of the assessment", "title": "Answer", "type": "boolean"}, "explanation": {"description": "Explanation of the binary answer in one or two sentences", "title": "Explanation", "type": "string"}}, "required": ["answer", "explanation"]}
```



In [ ]:
agent = create_agent(
    model=ChatOllama(model= "granite4:350m", temperature=0),
    system_prompt=system_prompt,
)

# 7. Define the user input (title + abstract)
title = "The Impact of Climate Change on Biodiversity"
abstract = """
Climate change is altering ecosystems worldwide, leading to shifts in species distributions.
This study explores how rising temperatures and changing precipitation patterns affect biodiversity.
"""

# 8. Invoke the agent with the user prompt
response = agent.invoke({
    "input": f"Title: {title}\nAbstract: {abstract}"
})
response

In [ ]:
# 9. Parse the response into a dictionary
# (Assuming the model returns a list of JSON objects, one per question)
response_text = response["messages"]

# Split the response into parts for each question
response_parts = response_text.strip().split("\n\n")

# Parse each part into the QuestionAnswer schema
answers_dict = {}
for q, question in enumerate(questions):
    question_key = question["name"]
    if q < len(response_parts):
        try:
            parsed = parser.parse(response_parts[q])
            answers_dict[question_key] = {
                "answer": parsed.answer,
                "explanation": parsed.explanation,
            }
        except:
            # Fallback if parsing fails
            answers_dict[question_key] = {
                "answer": "Unknown",
                "explanation": "Failed to parse response.",
            }

# 10. Print the results
for question_key, answer_data in answers_dict.items():
    print(f"{question_key}:")
    print(f"  Answer: {answer_data['answer']}")
    print(f"  Explanation: {answer_data['explanation']}\n")

AttributeError: 'list' object has no attribute 'strip'

In [ ]:
from typing import List, Dict
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain_core.output_parsers import PydanticOutputParser
from langchain_ollama import ChatOllama

# 1. Define the output schema for parsing
class QuestionAnswer(BaseModel):
    answer: str  # "Yes" or "No"
    explanation: str  # 1-2 sentences

# 2. Define the output parser
parser = PydanticOutputParser(pydantic_object=QuestionAnswer)

# 3. Define the list of questions as a dictionary
questions = [
    {"name": "Relevance", "question": "Is the abstract relevant to the title?"},
    {"name": "Clarity", "question": "Is the abstract clear and well-written?"},
    {"name": "Completeness", "question": "Does the abstract cover the main points?"},
]

# 4. Join the questions into a string for the system prompt
questions_str = "\n".join(
    [f"{q['name']}: {q['question']}" for q in questions]
)

# 5. Define the system prompt (static, no mustache)
system_prompt = f"""
You are a precise assistant.
For each of the following questions, answer with a JSON object containing:
- 'answer': 'Yes' or 'No'
- 'explanation': a short explanation (1-2 sentences)
"""

# 6. Create the agent with the static system prompt


{'messages': [AIMessage(content='{\n  "answer": "No",\n  "explanation": "The abstract is not relevant to the title, as it does not address the main content or purpose of the paper."\n}', additional_kwargs={}, response_metadata={'model': 'granite4:350m', 'created_at': '2026-06-24T15:26:30.019086298Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1012606349, 'load_duration': 62326719, 'prompt_eval_count': 255, 'prompt_eval_duration': 20384228, 'eval_count': 39, 'eval_duration': 894444923, 'logprobs': None, 'model_name': 'granite4:350m', 'model_provider': 'ollama'}, id='lc_run--019efa3d-688b-7552-b021-6383bf0b4aca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 255, 'output_tokens': 39, 'total_tokens': 294})]}

In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ToolStrategy
from typing import List, Dict, Any
import json

# Define the structured output schema
output_schema = {
    "type": "object",
    "properties": {
        "answer": {
            "type": "string",
            "enum": ["True", "False"],
            "description": "Binary answer: 'True' or 'False'"
        },
        "explanation": {
            "type": "string",
            "description": "A short explanation (1-2 sentences)"
        }
    },
    "required": ["answer", "explanation"]
}

# Initialize ChatOllama
model = ChatOllama(model="granite4:350m", temperature=0)

# Create the agent with structured output
agent = create_agent(
    model=model,
    response_format=ToolStrategy(output_schema),
    system_prompt="You are a precise assistant that analyse the tite and abstract of a paper to answer binary question. Answer with a JSON object containing 'answer' (True/False) and 'explanation' (1-2 sentences)."
)

# Function to process a list of questions
def assess_questions(questions: List[str]) -> List[Dict[str, Any]]:
    results = []
    for question in questions:
        response = agent.invoke({
            "messages": [{"role": "user", "content": question}]
        })
        # Extract the structured response
        structured_response = response.get("structured_response", {})
        results.append(structured_response)
    return results

# Example usage
questions = [
    "Title: The Impact of Climate Change on Biodiversity\nAbstract: Climate change is altering ecosystems worldwide, leading to shifts in species distributions. This study explores how rising temperatures and changing precipitation patterns affect biodiversity.",
    "Title: Quantum Computing in Finance\nAbstract: Quantum computing promises to revolutionize financial modeling by solving complex optimization problems faster than classical computers."
]

# Get results
results = assess_questions(questions)
for result in results:
    print(json.dumps(result, indent=2))

{
  "title": "The Impact of Climate Change on Biodiversity",
  "question": "Does this study focus on the impact of climate change on biodiversity?",
  "answer": "True",
  "explanation": "The study examines how climate change is altering ecosystems worldwide, leading to shifts in species distributions and thus affecting biodiversity."
}
{
  "title": "Quantum Computing in Finance",
  "question": "Is this study about the impact of climate change?",
  "answer": "True",
  "explanation": "The study focuses on quantum computing's potential to revolutionize financial modeling by solving complex optimization problems faster than classical computers, which could have significant implications for the field of finance."
}


In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ToolStrategy
from typing import List, Dict, Any
import json

# Define the structured output schema
output_schema = {
    "type": "object",
    "properties": {
        "answer": {
            "type": "string",
            "enum": ["True", "False"],
            "description": "Binary answer: 'True' or 'False'"
        },
        "explanation": {
            "type": "string",
            "description": "A short explanation (1-2 sentences)"
        }
    },
    "required": ["answer", "explanation"]
}

# Initialize ChatOllama
model = ChatOllama(model="granite4:350m", temperature=0)

# Create the agent with structured output
agent = create_agent(
    model=model,
    response_format=ToolStrategy(output_schema),
    system_prompt="You are a precise assistant that analyse the tite and abstract of a paper to answer binary question. Answer with a JSON object containing 'answer' (True/False) and 'explanation' (1-2 sentences)."
)

# Function to process a list of questions for multiple Title+Abstract pairs
def assess_questions(questions: List[Dict[str, str]], papers: List[Dict[str,str]]) -> List[Dict[str, Any]]:
    results = []
    for p in papers:
        title = p.get("title", "")
        print(title)
        abstract = p.get("abstract", "")
        for q in questions:
            print("  -  ", q)
            question = q.get("question", "")
            # Format the input for the agent
            input_text = f"Title: {title}\nAbstract: {abstract}\nQuestion: {question}"

            response = agent.invoke({
                "messages": [{"role": "user", "content": input_text}]
            })

            # Extract the structured response
            structured_response = response.get("structured_response", {})
            results.append({
                "title": title,
                "question": question,
                "question_name" : q.get("name"),
                **structured_response
            })
    return results

questions = [
{"name": "impact_climate_biodiversity", "question": "This papers talks about the impact of climate change on biodiversity"},
{"name": "impact_climate_human_scoiety", "question": "This papers talks about the impact of climate change on human society"}
]

# Example usage
papers = [
    {
        "title": "Quantum Computing in Finance",
        "abstract": "Quantum computing promises to revolutionize financial modeling by solving complex optimization problems faster than classical computers.",
    },
    {
        "title": "The Impact of Climate Change on Biodiversity",
        "abstract": "Climate change is altering ecosystems worldwide, leading to shifts in species distributions. This study explores how rising temperatures and changing precipitation patterns affect biodiversity.",
    },

]

# Get results
results = assess_questions(questions,papers)
for result in results:
    print(json.dumps(result, indent=2))

Quantum Computing in Finance
  -   {'name': 'impact_climate_biodiversity', 'question': 'This papers talks about the impact of climate change on biodiversity'}
